In [3]:
import rasterio  as rio
from pathlib import Path

In [9]:
YEARS = [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

BASE = Path("/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines")
INPUT = BASE / "sentinel_data/s2_indices/"
OUT = BASE / "sentinel_data/S2_indices_test/"

In [20]:
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling

# 1) Find one file per year
files = {}
for year in YEARS:
    in_folder = INPUT / str(year)
    print("Looking in:", in_folder, "| exists:", in_folder.exists())

    found = sorted(in_folder.glob(f"**east_*_ndsi_0.4.tiff"))
    if len(found) != 1:
        print(f"  Expected 1 file for {year}, found {len(found)}: {[f.name for f in found]}. Skipping.")
        continue
    files[year] = found[0]

# 2) Use the first year as the reference grid
first_file = next(iter(files.values()))
with rasterio.open(first_file) as ref:
    profile = ref.profile.copy()
    ref_transform = ref.transform
    ref_crs = ref.crs
    ref_shape = ref.shape

# 3) Fit every year onto that grid
img, valid_years = [], []
for year, f in files.items():
    arr = np.zeros(ref_shape, dtype=np.uint8)   # empty grid, filled with 0

    with rasterio.open(f) as src:
        reproject(
            source=rasterio.band(src, 1),
            destination=arr,
            dst_transform=ref_transform,
            dst_crs=ref_crs,
            resampling=Resampling.nearest,      # keeps values as 0/1
        )

    img.append(arr == 1)
    valid_years.append(year)

stack = np.stack(img)
print(stack.shape, valid_years)

Looking in: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/sentinel_data/s2_indices/2017 | exists: True
Looking in: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/sentinel_data/s2_indices/2018 | exists: True
Looking in: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/sentinel_data/s2_indices/2019 | exists: True
Looking in: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/sentinel_data/s2_indices/2020 | exists: True
Looking in: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/sentinel_data/s2_indices/2021 | exists: True
Looking in: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/sentinel_data/s2_indices/2022 | exists: True
Looking in: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/sentinel_data/s2_indices/2023 | exists: True
Looking in: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/sentinel_data/s2_indices/2024 | exists: True
Looking in: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/sentinel_data/s2_indices/2025 | exists: True
(9, 2378, 

In [21]:
idx = {year: i for i, year in enumerate(valid_years)}

def save_year(year, arr):
    out_file = OUT / str(year) / f"S2_mosaic_east_{year}_ndsi_0.4.tiff"
    out_file.parent.mkdir(parents=True, exist_ok=True)
    with rasterio.open(out_file, "w", **profile) as dst:
        dst.write(arr, 1)
    print(f"Saved filtered {year} to {out_file}")

# --- Block 1: 2019-2021, cascading (baseline = raw 2018) ---
prev = stack[idx[2017]]

for year in range(2018, 2022):
    target = stack[idx[year]]
    new_target = np.where((target == 1) & (prev == 1), 1, 0).astype(np.uint8)
    save_year(year, new_target)
    prev = new_target  # filtered result becomes the previous year

# --- Block 2: 2024, compared against the ORIGINAL 2023 ---
prev = stack[idx[2022]]

for year in range(2023, 2026):
    target = stack[idx[year]]
    new_target = np.where((target == 1) & (prev == 1), 1, 0).astype(np.uint8)
    save_year(year, new_target)
    prev = new_target  # filtered result becomes the previous year

Saved filtered 2018 to /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/sentinel_data/S2_indices_test/2018/S2_mosaic_east_2018_ndsi_0.4.tiff
Saved filtered 2019 to /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/sentinel_data/S2_indices_test/2019/S2_mosaic_east_2019_ndsi_0.4.tiff
Saved filtered 2020 to /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/sentinel_data/S2_indices_test/2020/S2_mosaic_east_2020_ndsi_0.4.tiff
Saved filtered 2021 to /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/sentinel_data/S2_indices_test/2021/S2_mosaic_east_2021_ndsi_0.4.tiff
Saved filtered 2023 to /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/sentinel_data/S2_indices_test/2023/S2_mosaic_east_2023_ndsi_0.4.tiff
Saved filtered 2024 to /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/sentinel_data/S2_indices_test/2024/S2_mosaic_east_2024_ndsi_0.4.tiff
Saved filtered 2025 to /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/sentinel_data/S2_indices_test/2025/S2_mosaic_east_2025